### Importing required libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import ipaddress
import pickle

### Mounting Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Importing the Dataset

In [3]:
dataset = pd.read_csv('/content/drive/MyDrive/sample.csv')

### Exploratory Data Analysis

In [4]:
# Checking out the first 5 rows in the data
dataset.head()

,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,payment_method,account_balance,transaction_status,ip_address,amt
0,TXN-3E3A44E74819,USR0020,7457.251963207814,2024-03-14T11:40:15.672605,JAI,ahmedabad,Travel,DEV-C894B8F5,mobile,Card,140227.79,success,192.96.196.31,NaN
1,TXN-84B8FC890498,USR0043,308.51678134343047,1717321977,Delhi,DEL,Travel,DEV-EC8BBC24,mobile,UPI,122548.77,success,192.138.55.249,NaN
2,TXN-95B71538FFAD,USR0044,₹3159,1710796175,jaipur,Jaipur,NaN,D??,web,Card,168910.49,success,192.73.41.151,NaN
3,TXN-657F2702B5B2,USR0060,5349.73,07/02/2024 03:50:23,BENGALURU,BENGALURU,Education,DEV-888653EA,web,UPI,9046.00,success,192.231.148.119,NaN
4,TXN-84C8BBF69256,USR0020,2454.5652128035144,"March 13, 2024 11:25 PM",jaipur,Jaipur,Utilities,DEV-C894B8F5,mobile,Card,153095.35,success,192.96.196.139,NaN


In [5]:
# Checking out missing values
dataset.isnull().sum()

,0
transaction_id,0
user_id,0
transaction_amount,73
transaction_timestamp,0
user_location,0
merchant_location,0
merchant_category,78
device_id,73
device_type,0
payment_method,74


In [6]:
# total number of duplicates in the data
dataset.duplicated().sum()

np.int64(21)

In [7]:
# dataset summary
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1447 entries, 0 to 1446
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   transaction_id         1447 non-null   object 
 1   user_id                1447 non-null   object 
 2   transaction_amount     1374 non-null   object 
 3   transaction_timestamp  1447 non-null   object 
 4   user_location          1447 non-null   object 
 5   merchant_location      1447 non-null   object 
 6   merchant_category      1369 non-null   object 
 7   device_id              1374 non-null   object 
 8   device_type            1447 non-null   object 
 9   payment_method         1373 non-null   object 
 10  account_balance        1373 non-null   float64
 11  transaction_status     1447 non-null   object 
 12  ip_address             1368 non-null   object 
 13  amt                    49 non-null     object 
dtypes: float64(1), object(13)
memory usage: 158.4+ KB


In [8]:
# numerical summary of the dataset
dataset.describe()

,account_balance
count,1373.000000
mean,73386.952403
std,55553.341123
min,0.000000
25%,23885.610000
50%,68661.030000
75%,121395.610000
max,191245.660000


**Summary of the Dataset**:

Total Number of Data Points: 1447

Total Number of Attributes: 14

Duplicate Values: 21




### Cleaning the Dataset

#### Cleaning the `transaction_id` column

In [9]:
dataset['is_duplicate_txn'] = (
    dataset.duplicated(subset=['transaction_id'], keep=False) &
    ~dataset.duplicated(keep=False)
).astype(int)

In [10]:
dataset['is_duplicate_txn'].value_counts()

,count
is_duplicate_txn,
0,1447


Two transactions having the same transaction ID might indicate that there has been a fraud.

#### Cleaning the `user_id`

In [11]:
# For user_id (You must do this BEFORE dropping user_id!)
user_profiles = dataset.groupby('user_id').agg(
    user_transaction_count=('user_id', 'count')
).reset_index()
dataset = dataset.merge(user_profiles, on='user_id', how='left')

#### Cleaning the `transaction_amount` column

In [12]:
# 1. Create the binary flag for missing values (captures the fraud signal)
missing_indicators = [None, np.nan, '', 'NaN', 'NULL', 'null', 'N/A']
dataset['is_amount_missing'] = dataset['transaction_amount'].isin(missing_indicators).astype(int)

# 2. Convert to string and remove common non-numeric characters without Regex
dataset['transaction_amount_clean'] = (
    dataset['transaction_amount']
    .astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)

# 3. Safely convert to numeric: turns unparseable text (like "abc") into NaN
dataset['transaction_amount_clean'] = pd.to_numeric(dataset['transaction_amount_clean'], errors='coerce')

# 4. Calculate the MEDIAN to avoid skewing from extreme 1.6M outliers
amt_median = dataset['transaction_amount_clean'].median()

# 5. Impute the NaNs with the robust median value
dataset['transaction_amount'] = dataset['transaction_amount_clean'].fillna(amt_median)

# 6. Cleanup: Drop temporary column and the redundant 'amt' column if it exists
dataset = dataset.drop(columns=['transaction_amount_clean', 'amt'], errors='ignore')

# 7. Verify the results
print(f"Median used for imputation: {amt_median:.2f}")
print("Remaining nulls in transaction_amount:", dataset['transaction_amount'].isnull().sum())

Median used for imputation: 3272.78
Remaining nulls in transaction_amount: 0


In [13]:
dataset.head()

,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,payment_method,account_balance,transaction_status,ip_address,is_duplicate_txn,user_transaction_count,is_amount_missing
0,TXN-3E3A44E74819,USR0020,7457.251963,2024-03-14T11:40:15.672605,JAI,ahmedabad,Travel,DEV-C894B8F5,mobile,Card,140227.79,success,192.96.196.31,0,23,0
1,TXN-84B8FC890498,USR0043,308.516781,1717321977,Delhi,DEL,Travel,DEV-EC8BBC24,mobile,UPI,122548.77,success,192.138.55.249,0,21,0
2,TXN-95B71538FFAD,USR0044,3159.000000,1710796175,jaipur,Jaipur,NaN,D??,web,Card,168910.49,success,192.73.41.151,0,21,0
3,TXN-657F2702B5B2,USR0060,5349.730000,07/02/2024 03:50:23,BENGALURU,BENGALURU,Education,DEV-888653EA,web,UPI,9046.00,success,192.231.148.119,0,19,0
4,TXN-84C8BBF69256,USR0020,2454.565213,"March 13, 2024 11:25 PM",jaipur,Jaipur,Utilities,DEV-C894B8F5,mobile,Card,153095.35,success,192.96.196.139,0,23,0


In [14]:
dataset['is_amount_outlier'] = (
    dataset['transaction_amount'] > dataset['transaction_amount'].quantile(0.99)
).astype(int)

#### Cleaning the `transaction_timestamp` column

In [15]:
dataset['transaction_timestamp'][0]

'2024-03-14T11:40:15.672605'

In [16]:
# Function to parse and standardize the mixed timestamps
def parse_timestamp(ts):
    if pd.isna(ts):
        return pd.NaT

    ts_str = str(ts).strip()

    # Check if it's a 10-digit Unix timestamp (seconds)
    if ts_str.isdigit() and len(ts_str) == 10:
        return pd.to_datetime(int(ts_str), unit='s')

    # Check if it's a 13-digit Unix timestamp (milliseconds)
    if ts_str.isdigit() and len(ts_str) == 13:
        return pd.to_datetime(int(ts_str), unit='ms')

    # Let Pandas infer and parse the remaining text/date formats
    try:
        return pd.to_datetime(ts_str, format='mixed')
    except Exception:
        return pd.NaT

# Apply the custom parsing function
dataset['transaction_timestamp'] = dataset['transaction_timestamp'].apply(parse_timestamp)

# Inspect the standardized formats
print(dataset['transaction_timestamp'].head())

0   2024-03-14 11:40:15.672605
1   2024-06-02 09:52:57.000000
2   2024-03-18 21:09:35.000000
3   2024-07-02 03:50:23.000000
4   2024-03-13 23:25:00.000000
Name: transaction_timestamp, dtype: datetime64[ns]


In [17]:
dataset['txn_month'] = dataset['transaction_timestamp'].dt.month
dataset['txn_day'] = dataset['transaction_timestamp'].dt.day
dataset['txn_hour'] = dataset['transaction_timestamp'].dt.hour
dataset['txn_minute'] = dataset['transaction_timestamp'].dt.minute

# Extracting categorical/behavioral components
dataset['txn_day_of_week'] = dataset['transaction_timestamp'].dt.dayofweek # 0=Monday, 6=Sunday
dataset['txn_is_weekend'] = (dataset['transaction_timestamp'].dt.dayofweek >= 5).astype(int)

In [18]:
dataset.head()

,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,payment_method,...,is_duplicate_txn,user_transaction_count,is_amount_missing,is_amount_outlier,txn_month,txn_day,txn_hour,txn_minute,txn_day_of_week,txn_is_weekend
0,TXN-3E3A44E74819,USR0020,7457.251963,2024-03-14 11:40:15.672605,JAI,ahmedabad,Travel,DEV-C894B8F5,mobile,Card,...,0,23,0,0,3,14,11,40,3,0
1,TXN-84B8FC890498,USR0043,308.516781,2024-06-02 09:52:57.000000,Delhi,DEL,Travel,DEV-EC8BBC24,mobile,UPI,...,0,21,0,0,6,2,9,52,6,1
2,TXN-95B71538FFAD,USR0044,3159.000000,2024-03-18 21:09:35.000000,jaipur,Jaipur,NaN,D??,web,Card,...,0,21,0,0,3,18,21,9,0,0
3,TXN-657F2702B5B2,USR0060,5349.730000,2024-07-02 03:50:23.000000,BENGALURU,BENGALURU,Education,DEV-888653EA,web,UPI,...,0,19,0,0,7,2,3,50,1,0
4,TXN-84C8BBF69256,USR0020,2454.565213,2024-03-13 23:25:00.000000,jaipur,Jaipur,Utilities,DEV-C894B8F5,mobile,Card,...,0,23,0,0,3,13,23,25,2,0


Since all the transactions have taken place in the year 2024 we will be dropping that column. We have also removed the second column because it will make the data very granular.

#### Cleaning the `user_location` and `merchant_location` columns:

In [19]:
# Checking for unique values in the locations
dataset['user_location'].value_counts()

,count
user_location,
JAI,98
jaipur,75
Jaipur,65
lucknow,58
Lucknow,57
...,...
Bangalo,1
De...,1
che,1


In [20]:
dataset['merchant_location'].value_counts()

,count
merchant_location,
Jaipur,87
jaipur,77
Lucknow,64
LKO,63
JAI,63
lucknow,53
Pune,47
CCU,43
ahmedabad,41


In [21]:
# 1. Basic cleaning: lowercasing and stripping whitespace
dataset['user_location'] = dataset['user_location'].str.lower().str.strip()
dataset['merchant_location'] = dataset['merchant_location'].str.lower().str.strip()

# 2. Define the map (using lowercase keys for consistency)
city_map = {
    # Mumbai
    'mumbai': 'Mumbai', 'bombay': 'Mumbai', 'bom': 'Mumbai', 'mumb...': 'Mumbai', 'm#': 'Mumbai',

    # Jaipur
    'jaipur': 'Jaipur', 'jai': 'Jaipur', 'j...': 'Jaipur', 'jai??': 'Jaipur',

    # Lucknow
    'lucknow': 'Lucknow', 'lko': 'Lucknow', 'lu#': 'Lucknow', 'luc#': 'Lucknow', 'l...': 'Lucknow', 'l??': 'Lucknow', 'luckn':'Lucknow', 'l':'Lucknow',

    # Chennai
    'chennai': 'Chennai', 'maa': 'Chennai', 'madras': 'Chennai', 'chenna#': 'Chennai', 'madr#' :'Chennai', 'che': 'Chennai',

    # Hyderabad
    'hyderabad': 'Hyderabad', 'hyd': 'Hyderabad', 'hyde': 'Hyderabad', 'hyde...': 'Hyderabad', 'hyderab#':'Hyderabad', 'hyder...':'Hyderabad',

    # Kolkata
    'kolkata': 'Kolkata', 'ccu': 'Kolkata', 'calcutta': 'Kolkata',

    # Delhi
    'delhi': 'Delhi', 'del': 'Delhi', 'new delhi': 'Delhi', 'h...': 'Hyderabad', 'de...':'Delhi',

    # Bangalore
    'bangalore': 'Bangalore', 'blr': 'Bangalore', 'bengaluru': 'Bangalore', 'beng...': 'Bangalore', 'bangalo': 'Bangalore', 'ba': 'Bangalore',

    # Ahmedabad
    'ahmedabad': 'Ahmedabad', 'amd': 'Ahmedabad',

    # Pune
    'pune': 'Pune', 'pnq': 'Pune', 'pu...':'Pune', 'pu??':'Pune', 'pun??':'Pune',

    # International
    'new york': 'New York', 'new yor#': 'New York',
    'dubai': 'Dubai',
    'bangkok': 'Bangkok',
    'singapore': 'Singapore',
    'international': 'Unknown',
}

# 3. Apply the replacement
dataset['user_location'] = dataset['user_location'].replace(city_map)
dataset['merchant_location'] = dataset['merchant_location'].replace(city_map)

# 4. Final Polish: Capitalize any remaining strings that weren't in the map
# (This handles names like "Patna" even if they weren't explicitly mapped)
dataset['user_location'] = dataset['user_location'].str.title()
dataset['merchant_location'] = dataset['merchant_location'].str.title()

print(dataset['user_location'].value_counts())
print(dataset['merchant_location'].value_counts())

user_location
Jaipur       241
Mumbai       188
Lucknow      176
Kolkata      139
Chennai      137
Hyderabad    135
Delhi        127
Bangalore    106
Pune         100
Ahmedabad     83
New York       5
Bangkok        4
Dubai          4
Singapore      2
Name: count, dtype: int64
merchant_location
Jaipur       227
Lucknow      180
Mumbai       174
Kolkata      143
Chennai      137
Hyderabad    135
Delhi        120
Bangalore    118
Pune         103
Ahmedabad     95
New York       5
Bangkok        4
Dubai          4
Singapore      2
Name: count, dtype: int64


In [22]:
# 1. Define your coordinate dictionary
# Ensure type hints align with Python 3.10 syntax
city_coords: dict[str, tuple[float, float]] = {
    # Format: "Standardized_City_Name": (Latitude, Longitude)

    # Domestic (India)
    'Mumbai': (19.0760, 72.8777),
    'Jaipur': (26.9124, 75.7873),
    'Lucknow': (26.8467, 80.9462),
    'Chennai': (13.0827, 80.2707),
    'Hyderabad': (17.3850, 78.4867),
    'Kolkata': (22.5726, 88.3639),
    'Delhi': (28.7041, 77.1025),
    'Bangalore': (12.9716, 77.5946),
    'Ahmedabad': (23.0225, 72.5714),
    'Pune': (18.5204, 73.8567),

    # International
    'New York': (40.7128, -74.0060),
    'Dubai': (25.2048, 55.2708),
    'Bangkok': (13.7563, 100.5018),
    'Singapore': (1.3521, 103.8198),

    # Fallback for truly unresolvable locations
    'Unknown': (float('nan'), float('nan'))
}

def apply_geospatial_encoding(df: pd.DataFrame, col_name: str) -> pd.DataFrame:
    """
    Takes a dataframe and a location column name, and creates two new
    columns for Latitude and Longitude based on the city_coords dictionary.
    """

    # Define the new column names based on the target column
    lat_col = f"{col_name}_lat"
    lon_col = f"{col_name}_lon"

    # Map the coordinates
    # We use .get() so that if a city is missing from the dictionary,
    # it safely assigns NaN instead of throwing a KeyError.
    df[lat_col] = df[col_name].apply(lambda city: city_coords.get(city, (np.nan, np.nan))[0])
    df[lon_col] = df[col_name].apply(lambda city: city_coords.get(city, (np.nan, np.nan))[1])

    return df

# Ensure merchant_location is also standardized before applying!
dataset = apply_geospatial_encoding(dataset, "user_location")
dataset = apply_geospatial_encoding(dataset, "merchant_location")

We will be using geospatial encoding to encode our locations.

In [23]:
dataset.head()

,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,payment_method,...,txn_month,txn_day,txn_hour,txn_minute,txn_day_of_week,txn_is_weekend,user_location_lat,user_location_lon,merchant_location_lat,merchant_location_lon
0,TXN-3E3A44E74819,USR0020,7457.251963,2024-03-14 11:40:15.672605,Jaipur,Ahmedabad,Travel,DEV-C894B8F5,mobile,Card,...,3,14,11,40,3,0,26.9124,75.7873,23.0225,72.5714
1,TXN-84B8FC890498,USR0043,308.516781,2024-06-02 09:52:57.000000,Delhi,Delhi,Travel,DEV-EC8BBC24,mobile,UPI,...,6,2,9,52,6,1,28.7041,77.1025,28.7041,77.1025
2,TXN-95B71538FFAD,USR0044,3159.000000,2024-03-18 21:09:35.000000,Jaipur,Jaipur,NaN,D??,web,Card,...,3,18,21,9,0,0,26.9124,75.7873,26.9124,75.7873
3,TXN-657F2702B5B2,USR0060,5349.730000,2024-07-02 03:50:23.000000,Bangalore,Bangalore,Education,DEV-888653EA,web,UPI,...,7,2,3,50,1,0,12.9716,77.5946,12.9716,77.5946
4,TXN-84C8BBF69256,USR0020,2454.565213,2024-03-13 23:25:00.000000,Jaipur,Jaipur,Utilities,DEV-C894B8F5,mobile,Card,...,3,13,23,25,2,0,26.9124,75.7873,26.9124,75.7873


#### Cleaning the `merchant_category` column

In [24]:
# 1. Fill missing values first!
dataset['merchant_category'] = dataset['merchant_category'].fillna("Unknown")

In [25]:
# Calculate how many times each user has shopped in each category
user_cat_counts = dataset.groupby(['user_id', 'merchant_category']).size().reset_index(name='cat_count')
user_total_counts = dataset.groupby('user_id').size().reset_index(name='total_count')

# Merge and calculate the percentage (affinity)
affinity_df = pd.merge(user_cat_counts, user_total_counts, on='user_id')
affinity_df['user_category_affinity'] = affinity_df['cat_count'] / affinity_df['total_count']

# Merge back to the main dataframe
dataset = pd.merge(dataset, affinity_df[['user_id', 'merchant_category', 'user_category_affinity']],
              on=['user_id', 'merchant_category'], how='left')

# Map category to the median transaction amount of that category
cat_median_amounts = dataset.groupby('merchant_category')['transaction_amount'].median().to_dict()
dataset['category_economic_weight'] = dataset['merchant_category'].map(cat_median_amounts)

In [26]:
dataset.head()

,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,payment_method,...,txn_hour,txn_minute,txn_day_of_week,txn_is_weekend,user_location_lat,user_location_lon,merchant_location_lat,merchant_location_lon,user_category_affinity,category_economic_weight
0,TXN-3E3A44E74819,USR0020,7457.251963,2024-03-14 11:40:15.672605,Jaipur,Ahmedabad,Travel,DEV-C894B8F5,mobile,Card,...,11,40,3,0,26.9124,75.7873,23.0225,72.5714,0.260870,3272.780000
1,TXN-84B8FC890498,USR0043,308.516781,2024-06-02 09:52:57.000000,Delhi,Delhi,Travel,DEV-EC8BBC24,mobile,UPI,...,9,52,6,1,28.7041,77.1025,28.7041,77.1025,0.238095,3272.780000
2,TXN-95B71538FFAD,USR0044,3159.000000,2024-03-18 21:09:35.000000,Jaipur,Jaipur,Unknown,D??,web,Card,...,21,9,0,0,26.9124,75.7873,26.9124,75.7873,0.095238,3794.400850
3,TXN-657F2702B5B2,USR0060,5349.730000,2024-07-02 03:50:23.000000,Bangalore,Bangalore,Education,DEV-888653EA,web,UPI,...,3,50,1,0,12.9716,77.5946,12.9716,77.5946,0.157895,3272.780000
4,TXN-84C8BBF69256,USR0020,2454.565213,2024-03-13 23:25:00.000000,Jaipur,Jaipur,Utilities,DEV-C894B8F5,mobile,Card,...,23,25,2,0,26.9124,75.7873,26.9124,75.7873,0.086957,3225.019154


#### Cleaning the `device_id` column

In [27]:
# Count how many unique users are associated with each device
users_per_device = dataset.groupby('device_id')['user_id'].nunique().to_dict()

# Map this count back to our main dataframe
dataset['unique_users_on_device'] = dataset['device_id'].map(users_per_device)

# If a device is missing (NaN), we just assume 1 user to be safe
dataset['unique_users_on_device'] = dataset['unique_users_on_device'].fillna(1)

In [28]:
dataset['is_unique_users_on_device_outlier'] = (
    dataset['unique_users_on_device'] > dataset['unique_users_on_device'].quantile(0.99)
).astype(int)

#### Cleaning the `device_type` column


In [29]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(drop="first")

# Converting device_type str to numerical data using one hot
device_type_encoded = encoder.fit_transform(dataset[["device_type"]]).toarray()

# ADDED: index=dataset.index to ensure rows align perfectly
encoded_df = pd.DataFrame(
    device_type_encoded,
    columns=encoder.get_feature_names_out(["device_type"]),
    index=dataset.index
)
dataset = pd.concat([dataset, encoded_df], axis=1)

#### Cleaning the `payment_method` column

In [30]:
# Converting payment_method to numerical value by get dummies
dataset=pd.get_dummies(dataset,columns=['payment_method'],prefix='pay',drop_first=True,dtype=int)

#### Cleaning the `account_balance` column

In [31]:
# 1. Flag the missing balances (The Fraud Signal)
# We still want to remember WHICH rows were missing, as this is a strong fraud indicator
dataset['is_balance_missing'] = dataset['account_balance'].isnull().astype(int)

# 2. Force the column to be numeric, converting any text junk to NaN
dataset['account_balance'] = pd.to_numeric(dataset['account_balance'], errors='coerce')

# 3. Calculate the median and fill the missing values
# We calculate the median BEFORE filling so we don't skew our own math!
balance_median = dataset['account_balance'].median()
dataset['account_balance'] = dataset['account_balance'].fillna(balance_median)

# 4. Engineer the "Balance Utilization" Feature
# We use numpy's select() to handle edge cases cleanly
conditions = [
    dataset['transaction_amount'] < 0,  # Condition A: Transaction amount is negative (e.g., Refunds)
    dataset['account_balance'] < 0      # Condition B: Account balance is negative (Legitimate overdrafts)
]

choices = [
    -1.0,  # Choice A: Set utilization to -1.0 if the transaction is negative
    -1.0   # Choice B: Set utilization to -1.0 if the balance is overdrawn
]

# Default calculation for normal transactions:
# We still add +0.01 (epsilon) to the balance to prevent a "Division by Zero" crash
# just in case a user has an account balance of exactly 0.00
default_choice = dataset['transaction_amount'] / (dataset['account_balance'] + 0.01)

dataset['balance_utilization'] = np.select(conditions, choices, default=default_choice)

# Quick sanity check on the new feature
print(f"Median balance used for imputation: {balance_median:.2f}")
print("Missing balances flagged:", dataset['is_balance_missing'].sum())
print("Negative Utilizations (-1.0):", (dataset['balance_utilization'] == -1.0).sum())

Median balance used for imputation: 68661.03
Missing balances flagged: 74
Negative Utilizations (-1.0): 0


In [32]:
dataset.head()

,transaction_id,user_id,transaction_amount,transaction_timestamp,user_location,merchant_location,merchant_category,device_id,device_type,account_balance,...,category_economic_weight,unique_users_on_device,is_unique_users_on_device_outlier,device_type_mobile,device_type_web,pay_NetBanking,pay_UPI,pay_Wallet,is_balance_missing,balance_utilization
0,TXN-3E3A44E74819,USR0020,7457.251963,2024-03-14 11:40:15.672605,Jaipur,Ahmedabad,Travel,DEV-C894B8F5,mobile,140227.79,...,3272.780000,1.0,0,1.0,0.0,0,0,0,0,0.053180
1,TXN-84B8FC890498,USR0043,308.516781,2024-06-02 09:52:57.000000,Delhi,Delhi,Travel,DEV-EC8BBC24,mobile,122548.77,...,3272.780000,1.0,0,1.0,0.0,0,1,0,0,0.002518
2,TXN-95B71538FFAD,USR0044,3159.000000,2024-03-18 21:09:35.000000,Jaipur,Jaipur,Unknown,D??,web,168910.49,...,3794.400850,3.0,1,0.0,1.0,0,0,0,0,0.018702
3,TXN-657F2702B5B2,USR0060,5349.730000,2024-07-02 03:50:23.000000,Bangalore,Bangalore,Education,DEV-888653EA,web,9046.00,...,3272.780000,1.0,0,0.0,1.0,0,1,0,0,0.591391
4,TXN-84C8BBF69256,USR0020,2454.565213,2024-03-13 23:25:00.000000,Jaipur,Jaipur,Utilities,DEV-C894B8F5,mobile,153095.35,...,3225.019154,1.0,0,1.0,0.0,0,0,0,0,0.016033


#### Cleaning the `transaction_status` column

In [33]:
# Finding the values count in transaction_status
dataset["transaction_status"].value_counts()

# Using the dummies function to convert the transaction_status columns to numerical values
# Added drop_first=True to drop the first encoded column
dataset = pd.get_dummies(dataset, columns=['transaction_status'], prefix="transaction_status", drop_first=True, dtype=int)

#### Cleaning the `ip_address` column

In [34]:
def check_ip_suspicious(ip):
    # If it is completely missing (NaN), it is suspicious
    if pd.isna(ip) or ip in ["", "NA", "$N/A$", "nan", "NULL"]:
        return 1 # Flag as suspicious (1)

    ip_str = str(ip).strip()

    # Ask Python's official networking library if the IP is mathematically valid
    try:
        ipaddress.ip_address(ip_str)
        return 0 # It is valid, not suspicious (0)
    except ValueError:
        return 1 # It failed validation! Flag as suspicious (1)

# 1. Engineer the "Suspicious IP" Fraud Feature
dataset['is_ip_suspicious'] = dataset['ip_address'].apply(check_ip_suspicious)

#### Cleaning the `amt` column

There is nothing to play in the amount column as we have already used it.

### Exporting the Cleaned Dataset

In [35]:
dataset.to_csv('cleaned_dataset.csv')